# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

Google Colab starts with a fresh runtime, so this notebook reloads the repo and dataset instead of depending on variables from previous weeks.

In [21]:
# Clone the repository
!git clone https://github.com/martindiarua/ML_01.git
# Move to the raw data folder
%cd ML_01/data/raw

Cloning into 'ML_01'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 223 (delta 107), reused 111 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (223/223), 3.30 MiB | 12.61 MiB/s, done.
Resolving deltas: 100% (107/107), done.
/content/ML_01/ML_01/data/raw


In [22]:
import os
import glob
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

print("Current folder:", os.getcwd())
print("\nFiles in data/raw:")
for file in glob.glob("*"):
    print(file)
# Find CSV files in the raw data folder
csv_files = glob.glob("*.csv")

print("CSV files found:", csv_files)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file was found in data/raw.")

# Use the first CSV file found
data_path = csv_files[0]

df = pd.read_csv(data_path)

print("Loaded:", data_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

# Recreate the target used in the previous modeling work.
# CTR and engagement_rate are deliberately NOT used as model features
# because they were used to construct this target.

df["refresh_priority"] = (
    (df["ctr"] < 0.05) &
    (df["engagement_rate"] < 0.40)
).astype(int)

print(df["refresh_priority"].value_counts())

Current folder: /content/ML_01/ML_01/data/raw

Files in data/raw:
content_refresh_anonymized.csv
CSV files found: ['content_refresh_anonymized.csv']
Loaded: content_refresh_anonymized.csv
Rows: 30000
Columns: 44
refresh_priority
0    17038
1    12962
Name: count, dtype: int64


In [23]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

missing_features = [col for col in features if col not in df.columns]

if missing_features:
    raise KeyError(f"Missing expected features: {missing_features}")

print("Features used:", features)
print("Number of features:", len(features))

Features used: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'trend_pct']
Number of features: 24


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The goal here is to turn the model output into a queue a content team can actually use.

The model gives each page a probability of being in the `refresh_priority` class. I use that probability to rank the pages from higher to lower priority.

I am not treating the score as proof that a page needs a refresh. It is only a decision-support signal.

I also add simple reason codes so a reviewer can see why a page deserves attention. The reasons are based on the page's observable signals, such as high staleness, weak recent trend, or high search demand.

The final action is still decided by a person.

In [24]:
X = df[features].copy()
y = df["refresh_priority"]
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())
print(
    "Shared clients:",
    len(
        set(df.iloc[train_idx]["client_id"])
        & set(df.iloc[test_idx]["client_id"])
    )
)

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Shared clients: 0


In [25]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.50).astype(int)

print("Test F1:", round(f1_score(y_test, test_prediction), 4))
print("Test accuracy:", round(accuracy_score(y_test, test_prediction), 4))

Test F1: 0.9218
Test accuracy: 0.9179


The validation model above is kept separate from the queue model.

For the final queue, I retrain the same model on all available rows. This does not create a new target or a new modeling method. It simply allows the final scoring step to use all of the available data.

In [26]:
queue_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

queue_model.fit(X, y)

df["priority_score"] = queue_model.predict_proba(X)[:, 1]

df[["content_id", "priority_score"]].head()

,content_id,priority_score
0,content_304f48230142,6.029200e-13
1,content_a1fb4e703a9e,1.726695e-01
2,content_9aa793d4d895,9.662084e-03
3,content_331d6c4de07b,1.243969e-26
4,content_d99b7a2d90ca,5.877802e-08


### Reason codes

I am keeping the reason codes simple.

- `STALE` = the page has gone a long time without an update.
- `DECLINING` = the recent trend is negative.
- `HIGH_DEMAND` = the page has meaningful search demand.
- `VISIBLE` = the page already has search impressions, so there is something measurable to improve.
- `MULTIPLE_SIGNALS` = more than one of these conditions is present.

These are not explanations of why the page is underperforming. They are just useful review cues.

In [27]:
# Use percentile-based thresholds so the rules adapt to this dataset.

stale_threshold = df["days_since_last_update"].quantile(0.75)
demand_threshold = df["search_volume"].quantile(0.75)
impression_threshold = df["impressions_90d"].quantile(0.50)

df["reason_codes"] = ""

def build_reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= stale_threshold:
        reasons.append("STALE")

    if row["trend_pct"] < 0:
        reasons.append("DECLINING")

    if row["search_volume"] >= demand_threshold:
        reasons.append("HIGH_DEMAND")

    if row["impressions_90d"] >= impression_threshold:
        reasons.append("VISIBLE")

    if len(reasons) >= 2:
        reasons.append("MULTIPLE_SIGNALS")

    return ", ".join(reasons) if reasons else "REVIEW"

df["reason_codes"] = df.apply(build_reason_codes, axis=1)

print("Stale threshold:", round(stale_threshold, 2))
print("Demand threshold:", round(demand_threshold, 2))
print("Visible threshold:", round(impression_threshold, 2))

Stale threshold: 104.0
Demand threshold: 20.0
Visible threshold: 731.0


In [28]:
def assign_action(row):
    reasons = row["reason_codes"]

    if "STALE" in reasons and "DECLINING" in reasons:
        return "Refresh and review"

    if "DECLINING" in reasons and "HIGH_DEMAND" in reasons:
        return "Investigate and refresh"

    if "STALE" in reasons:
        return "Freshness review"

    if "HIGH_DEMAND" in reasons and "VISIBLE" in reasons:
        return "Optimization review"

    if "VISIBLE" in reasons:
        return "Content review"

    return "Monitor"

df["recommended_action"] = df.apply(assign_action, axis=1)

In [29]:
queue_columns = [
    "content_id",
    "client_id",
    "priority_score",
    "recommended_action",
    "reason_codes",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "trend_pct",
    "days_since_last_update",
    "word_count",
    "content_age_days"
]

action_queue = (
    df[queue_columns]
    .sort_values("priority_score", ascending=False)
    .reset_index(drop=True)
)

action_queue.insert(
    0,
    "priority_rank",
    range(1, len(action_queue) + 1)
)

action_queue.head(20)

,priority_rank,content_id,client_id,priority_score,recommended_action,reason_codes,search_volume,impressions_90d,clicks_90d,trend_pct,days_since_last_update,word_count,content_age_days
0,1,content_c8e9d6ab9013,client_19581e27de,1.000000,Refresh and review,"STALE, DECLINING, HIGH_DEMAND, VISIBLE, MULTIP...",20.0,208678,0,-43.4,104,NaN,362
1,2,content_39881853ef0c,client_f369cb89fc,1.000000,Investigate and refresh,"DECLINING, HIGH_DEMAND, VISIBLE, MULTIPLE_SIGNALS",170.0,112434,10,-42.0,20,2810.0,97
2,3,content_453722754fea,client_f369cb89fc,1.000000,Content review,"DECLINING, VISIBLE, MULTIPLE_SIGNALS",10.0,140079,16,-52.9,20,2700.0,97
3,4,content_54baba704595,client_6208ef0f77,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,130617,8,-54.8,104,5813.0,286
4,5,content_fb4bf6555c79,client_6208ef0f77,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,84093,3,-83.0,104,6681.0,299
5,6,content_e752a4e03dd3,client_6208ef0f77,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,130892,15,-52.7,104,6442.0,287
6,7,content_109f8f7c9d39,client_6208ef0f77,1.000000,Freshness review,"STALE, VISIBLE, MULTIPLE_SIGNALS",0.0,90476,6,126.8,104,7295.0,299
7,8,content_65114d89496d,client_19581e27de,1.000000,Content review,"DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,72631,12,-74.8,22,NaN,482
8,9,content_e5f459e737b7,client_f369cb89fc,0.999997,Investigate and refresh,"DECLINING, HIGH_DEMAND, VISIBLE, MULTIPLE_SIGNALS",30.0,56363,3,-27.2,20,2916.0,147
9,10,content_6e28a04c07a8,client_6208ef0f77,0.999977,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,41226,2,-39.0,104,6367.0,139


In [30]:
print("Total pages:", len(action_queue))
print("\nRecommended action counts:")
print(action_queue["recommended_action"].value_counts())

print("\nTop 10 priority pages:")
display(action_queue.head(10))

Total pages: 30000

Recommended action counts:
recommended_action
Monitor                    10193
Refresh and review          6905
Content review              5661
Investigate and refresh     4074
Freshness review            2186
Optimization review          981
Name: count, dtype: int64

Top 10 priority pages:


,priority_rank,content_id,client_id,priority_score,recommended_action,reason_codes,search_volume,impressions_90d,clicks_90d,trend_pct,days_since_last_update,word_count,content_age_days
0,1,content_c8e9d6ab9013,client_19581e27de,1.000000,Refresh and review,"STALE, DECLINING, HIGH_DEMAND, VISIBLE, MULTIP...",20.0,208678,0,-43.4,104,NaN,362
1,2,content_39881853ef0c,client_f369cb89fc,1.000000,Investigate and refresh,"DECLINING, HIGH_DEMAND, VISIBLE, MULTIPLE_SIGNALS",170.0,112434,10,-42.0,20,2810.0,97
2,3,content_453722754fea,client_f369cb89fc,1.000000,Content review,"DECLINING, VISIBLE, MULTIPLE_SIGNALS",10.0,140079,16,-52.9,20,2700.0,97
3,4,content_54baba704595,client_6208ef0f77,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,130617,8,-54.8,104,5813.0,286
4,5,content_fb4bf6555c79,client_6208ef0f77,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,84093,3,-83.0,104,6681.0,299
5,6,content_e752a4e03dd3,client_6208ef0f77,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,130892,15,-52.7,104,6442.0,287
6,7,content_109f8f7c9d39,client_6208ef0f77,1.000000,Freshness review,"STALE, VISIBLE, MULTIPLE_SIGNALS",0.0,90476,6,126.8,104,7295.0,299
7,8,content_65114d89496d,client_19581e27de,1.000000,Content review,"DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,72631,12,-74.8,22,NaN,482
8,9,content_e5f459e737b7,client_f369cb89fc,0.999997,Investigate and refresh,"DECLINING, HIGH_DEMAND, VISIBLE, MULTIPLE_SIGNALS",30.0,56363,3,-27.2,20,2916.0,147
9,10,content_6e28a04c07a8,client_6208ef0f77,0.999977,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",0.0,41226,2,-39.0,104,6367.0,139


### What the queue means

The queue gives the content team a starting order for review.

A high score means the model sees the page as more similar to pages in the `refresh_priority` class. The reason codes add some context around the page.

This does not mean that the first page in the queue is automatically the first page that should be changed. The queue still needs human review.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The main use of this output is to help a content team decide which pages to review first.

It can be useful when there are many pages and it is difficult to manually check everything at once. The score gives a ranking, while the reason codes give a quick idea of what to look at.

The output is decision-support, not an automatic content strategy.

The model was trained on this dataset and validated with clients separated between train and test. That gives a more honest estimate than randomly mixing rows from the same clients.

There are still limits. The target was engineered from CTR and engagement rate, so those fields were removed from the model features to avoid leakage. The model therefore does not directly use the two measurements that created the label.

The model also shows association in this dataset. It does not prove that refreshing a page will improve traffic, rankings, or engagement.

It should not be treated as a production system or a guaranteed ranking formula.

In [31]:
print("Priority score summary:")
print(df["priority_score"].describe())

high_priority_cutoff = df["priority_score"].quantile(0.90)

high_priority_pages = df[
    df["priority_score"] >= high_priority_cutoff
]

print(
    "90th percentile score cutoff:",
    round(high_priority_cutoff, 4)
)

print(
    "Pages in the top 10% of model scores:",
    len(high_priority_pages)
)

Priority score summary:
count    30000.000000
mean         0.432039
std          0.390033
min          0.000000
25%          0.000263
50%          0.449013
75%          0.843613
max          1.000000
Name: priority_score, dtype: float64
90th percentile score cutoff: 0.9268
Pages in the top 10% of model scores: 3000


### Intended use

A reasonable workflow would be:

1. Use the model to rank pages.
2. Start with the highest-priority pages.
3. Check the reason codes.
4. Open the page and review the actual content.
5. Check search intent and current SERP position.
6. Decide whether a refresh is actually needed.
7. Record what action was taken.
8. Measure the page again after the change.

The model should help reduce the amount of manual searching needed to find candidates. It should not remove the human decision.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### **Human review**
A person must review a page before any content change is made.

The model only sees the structured data available in the dataset. It does not know whether the page is factually correct, whether the search intent has changed, whether the topic is strategically important, or whether a refresh would actually make business sense.

Before acting on a recommendation, the reviewer should check:

- Does the page still match the target search intent?
- Is the information outdated?
- Is the content actually thin or incomplete?
- Has the page lost visibility recently?
- Is there evidence of cannibalization?
- Is the topic still important to the business?
- Would updating the page improve the user experience?
- Are there legal, medical, financial, or other high-risk claims that need specialist review?

The model should never make these decisions by itself.

### **No-go list**

The following should **not** be automated from this model:

- Automatically rewriting or publishing content.
- Automatically deleting pages.
- Automatically changing important business claims.
- Automatically changing medical, legal, or financial information.
- Automatically deciding that a page has poor quality.
- Automatically changing the target keyword or search intent.
- Automatically merging or redirecting pages.
- Automatically declaring that a refresh caused a traffic increase.
- Automatically spending money based on the priority score.
- Automatically treating a high score as proof that a page will improve after a refresh.

The model can recommend where to look. A human decides what to do.

In [32]:
review_columns = [
    "priority_rank",
    "content_id",
    "priority_score",
    "recommended_action",
    "reason_codes"
]

review_queue = action_queue[review_columns].head(50).copy()

review_queue["human_reviewed"] = False
review_queue["action_approved"] = False
review_queue["review_notes"] = ""

display(review_queue.head(20))

,priority_rank,content_id,priority_score,recommended_action,reason_codes,human_reviewed,action_approved,review_notes
0,1,content_c8e9d6ab9013,1.000000,Refresh and review,"STALE, DECLINING, HIGH_DEMAND, VISIBLE, MULTIP...",False,False,
1,2,content_39881853ef0c,1.000000,Investigate and refresh,"DECLINING, HIGH_DEMAND, VISIBLE, MULTIPLE_SIGNALS",False,False,
2,3,content_453722754fea,1.000000,Content review,"DECLINING, VISIBLE, MULTIPLE_SIGNALS",False,False,
3,4,content_54baba704595,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",False,False,
4,5,content_fb4bf6555c79,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",False,False,
5,6,content_e752a4e03dd3,1.000000,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",False,False,
6,7,content_109f8f7c9d39,1.000000,Freshness review,"STALE, VISIBLE, MULTIPLE_SIGNALS",False,False,
7,8,content_65114d89496d,1.000000,Content review,"DECLINING, VISIBLE, MULTIPLE_SIGNALS",False,False,
8,9,content_e5f459e737b7,0.999997,Investigate and refresh,"DECLINING, HIGH_DEMAND, VISIBLE, MULTIPLE_SIGNALS",False,False,
9,10,content_6e28a04c07a8,0.999977,Refresh and review,"STALE, DECLINING, VISIBLE, MULTIPLE_SIGNALS",False,False,


### Archetype → action mapping

The earlier analysis showed that content can fall into different states based on age, freshness, visibility, and depth. I would use those states as review guides rather than fixed personas.

| Content state | Suggested human action |
|---|---|
| Old + stale + declining | Review for a possible refresh |
| Old + still visible | Check whether a targeted refresh could protect existing visibility |
| Thin + visible | Review missing sections before adding unnecessary length |
| High demand + declining | Investigate the page first because the opportunity may be meaningful |
| Fresh + performing well | Usually monitor rather than rewrite |
| Low visibility + low demand | Do not automatically spend refresh effort here |

These are review rules, not guarantees.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations can become less useful if the underlying data changes.

I would monitor both the model inputs and the output. The main things to watch are:

- the distribution of priority scores;
- the percentage of pages receiving high-priority scores;
- changes in important input fields;
- model performance when new outcomes become available;
- whether the recommended pages are actually useful to reviewers.

Retraining should not happen just because some time has passed. There should be a reason to believe the data or the decision problem has changed.

In [33]:
monitoring_summary = pd.DataFrame({
    "metric": [
        "Rows scored",
        "Mean priority score",
        "Median priority score",
        "90th percentile priority score",
        "High-priority pages",
        "High-priority share"
    ],
    "value": [
        len(df),
        df["priority_score"].mean(),
        df["priority_score"].median(),
        df["priority_score"].quantile(0.90),
        len(df[df["priority_score"] >= high_priority_cutoff]),
        (
            len(df[df["priority_score"] >= high_priority_cutoff])
            / len(df)
        )
    ]
})

monitoring_summary

,metric,value
0,Rows scored,30000.000000
1,Mean priority score,0.432039
2,Median priority score,0.449013
3,90th percentile priority score,0.926846
4,High-priority pages,3000.000000
5,High-priority share,0.100000


### Retrain triggers

I would consider retraining when:

1. New labeled data becomes available.
2. The model's F1 score falls clearly on a new validation set.
3. The distribution of important features changes a lot.
4. The priority score distribution changes sharply.
5. The content portfolio changes enough that the old training data no longer represents the current pages.
6. The target definition changes.

If the label definition changes, I would not simply retrain the old model. I would revisit the whole modeling setup because the prediction problem itself has changed.

For this project, monitoring is more important than setting an arbitrary monthly retraining schedule.

In [34]:
monitor_features = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

drift_snapshot = df[monitor_features].describe().T[
    ["mean", "std", "min", "50%", "max"]
]

drift_snapshot

,mean,std,min,50%,max
search_volume,158.882391,1518.270825,0.0,10.0,74000.0
impressions_90d,5200.366300,16838.019547,1.0,731.0,517715.0
clicks_90d,16.097333,75.076958,0.0,1.0,4178.0
sessions_90d,37.066633,107.069131,1.0,7.0,4345.0
content_age_days,256.167800,132.707930,90.0,236.0,564.0
days_since_last_update,46.098300,42.078709,1.0,20.0,373.0
trend_pct,-4.785969,473.861780,-100.0,-33.5,44900.0


### Cost and value thinking

The queue is most useful when the cost of reviewing a page is smaller than the possible value of finding a good optimization opportunity.

I would therefore review high-priority pages with meaningful existing visibility before spending time on pages that have little measurable demand.

This does not mean low-traffic pages have no value. It only means the model has less evidence to work with.

The practical goal is not to refresh the largest number of pages. It is to help the team spend its limited review time on pages where there is enough evidence to investigate.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The final queue is exported to `work/outputs/`.

The main file contains the page ranking, model score, reason codes, recommended action, and useful page-level signals.

This file is intended to become one of the inputs for the recommendations section of the final research paper.

In [35]:
%cd /content/ML_01

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

print("Output folder:", os.path.abspath(output_dir))

/content/ML_01
Output folder: /content/ML_01/work/outputs


In [36]:
queue_path = os.path.join(
    output_dir,
    "content_action_queue.csv"
)

action_queue.to_csv(
    queue_path,
    index=False
)

print("Saved:", queue_path)
print("Rows exported:", len(action_queue))

Saved: work/outputs/content_action_queue.csv
Rows exported: 30000


In [37]:
top_queue_path = os.path.join(
    output_dir,
    "top_50_content_action_queue.csv"
)

action_queue.head(50).to_csv(
    top_queue_path,
    index=False
)

print("Saved:", top_queue_path)

Saved: work/outputs/top_50_content_action_queue.csv


In [38]:
monitor_path = os.path.join(
    output_dir,
    "action_playbook_monitoring_summary.csv"
)

monitoring_summary.to_csv(
    monitor_path,
    index=False
)

print("Saved:", monitor_path)

Saved: work/outputs/action_playbook_monitoring_summary.csv


In [39]:
summary_path = os.path.join(
    output_dir,
    "action_playbook_summary.txt"
)

summary_text = f"""
Content Action Playbook Summary

Pages scored: {len(action_queue):,}

Top 10% priority cutoff: {high_priority_cutoff:.4f}

Pages in top 10%: {len(high_priority_pages):,}

Mean priority score: {df["priority_score"].mean():.4f}

Recommended actions:
{action_queue["recommended_action"].value_counts().to_string()}

The model output is decision-support only. Pages should be reviewed by a human before any content change is made.
"""

with open(summary_path, "w") as f:
    f.write(summary_text)

print("Saved:", summary_path)

Saved: work/outputs/action_playbook_summary.txt


## Final takeaway

The model gives the content team a ranked list of pages that deserve a closer look. The reason codes make the queue easier to understand by showing signals such as staleness, declining trend, search demand, and existing visibility.

The important point is that the score is not a final answer. A high-priority page still needs a human review before any refresh is approved.

The strongest use of this system is therefore as a decision-support tool: use the model to narrow the list, use the page data to understand the situation, and use human judgment to decide the actual content action.

The earlier analysis also suggests that older or declining pages with existing visibility are better candidates for investigation than pages with little measurable demand. This is consistent with the broader age, freshness, and visibility patterns observed in the dataset, but it does not prove that refreshing a page will cause better performance.

The playbook should therefore be used to prioritize review time, not to automate content decisions.

## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.